In [12]:
!pip install rank-bm25 sentence-transformers groq google-generativeai -q

In [13]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [14]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai
from groq import Groq

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [15]:
CORPUS = [
    "Quantum computing utilizes principles of quantum mechanics like superposition and entanglement to perform computations.",
    "A qubit, the basic unit of quantum information, can exist in a superposition of 0 and 1 simultaneously.",
    "Entanglement allows qubits to become correlated, where the state of one instantly influences the state of another, regardless of distance.",
    "Quantum algorithms, such as Shor's algorithm for factoring large numbers, offer exponential speedups over classical algorithms for specific problems.",
    "Cryogenic temperatures are often required for superconducting qubits to maintain their quantum coherence.",
    "Quantum key distribution (QKD) leverages quantum mechanics to establish provably secure communication channels.",
    "Annealing is a quantum computing paradigm used for optimization problems, differing from gate-based quantum computers.",
    "Noise in quantum systems leads to decoherence, a major challenge in building robust quantum computers.",
    "Quantum machine learning explores how quantum computers can accelerate machine learning tasks and improve model performance."
]

print(f"Corpus size: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i:02d}] {doc[:80]}...")

Corpus size: 9 documents
  [00] Quantum computing utilizes principles of quantum mechanics like superposition an...
  [01] A qubit, the basic unit of quantum information, can exist in a superposition of ...
  [02] Entanglement allows qubits to become correlated, where the state of one instantl...
  [03] Quantum algorithms, such as Shor's algorithm for factoring large numbers, offer ...
  [04] Cryogenic temperatures are often required for superconducting qubits to maintain...
  [05] Quantum key distribution (QKD) leverages quantum mechanics to establish provably...
  [06] Annealing is a quantum computing paradigm used for optimization problems, differ...
  [07] Noise in quantum systems leads to decoherence, a major challenge in building rob...
  [08] Quantum machine learning explores how quantum computers can accelerate machine l...


In [16]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.corpus_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)

    def _bm25_ranked(self, query: str) -> list[tuple[int, float]]:
        scores = self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked

    def _sbert_ranked(self, query: str) -> list[tuple[int, float]]:
        q_emb = self.sbert.encode([query], convert_to_numpy=True)
        corpus_norm = self.corpus_embeddings / (
            np.linalg.norm(self.corpus_embeddings, axis=1, keepdims=True) + 1e-9
        )
        q_norm = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
        sims = (corpus_norm @ q_norm.T).squeeze()
        ranked = sorted(enumerate(sims.tolist()), key=lambda x: x[1], reverse=True)
        return ranked

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_ranked  = self._bm25_ranked(query)
        sbert_ranked = self._sbert_ranked(query)

        bm25_rank  = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(bm25_ranked)}
        sbert_rank = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(sbert_ranked)}

        all_ids = set(range(len(self.corpus)))
        rrf_scores = {}
        for doc_id in all_ids:
            r_bm25  = bm25_rank.get(doc_id, len(self.corpus))
            r_sbert = sbert_rank.get(doc_id, len(self.corpus))
            rrf_scores[doc_id] = (
                1 / (self.k + r_bm25) +
                1 / (self.k + r_sbert)
            )

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        results = []
        for doc_id, score in sorted_docs[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(score, 6),
                "bm25_rank":  bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results

In [20]:
retriever = HybridRetriever(CORPUS)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)  # returns numpy array

    for cand, score in zip(candidates, scores):
        cand["ce_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

def hyde_expand(user_query: str) -> str:
    prompt = (
        "You are an expert in Quantum Computing. "
        "Write a concise, technical 2–3 sentence passage that directly answers "
        "the following question. Write as if it were an excerpt from a textbook.\n\n"
        f"Question: {user_query}\n\nPassage:"
    )
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.0)
    )
    return response.text.strip()

In [ ]:
def advanced_rag(user_query: str, verbose: bool = True) -> str:
    if verbose: print(f"\n{'='*60}\n Query: {user_query}")
    hypothetical_doc = hyde_expand(user_query)
    if verbose: print(f"\n HyDE doc:\n   {hypothetical_doc[:120]}...")

    candidates = retriever.retrieve(hypothetical_doc, top_k=6)
    if verbose:
        print(f"\n Hybrid retrieval top-3 (of 6 candidates):")
        for c in candidates[:3]:
            print(f"   RRF={c['rrf_score']} | {c['text'][:75]}")

    top_docs = rerank(user_query, candidates, top_k=3)
    if verbose:
        print(f"\n  After re-ranking:")
        for i, d in enumerate(top_docs):
            print(f"   #{i+1} CE={d['ce_score']:>7.4f} | {d['text'][:75]}")

    context = "\n".join(f"[{i+1}] {d['text']}" for i, d in enumerate(top_docs))
    prompt = (
        f"You are a helpful university AI assistant. "
        f"Answer the student's question using only the provided context. "
        f"Be concise and technical.\n\n"
        f"Context:\n{context}\n\n"
        f"Student question: {user_query}\n\n"
        f"Answer:"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=300,
    )
    answer = response.choices[0].message.content.strip()
    if verbose: print(f"\nAnswer:\n{answer}\n{'='*60}")
    return answer


_ = advanced_rag("what is quantum superposition?")


 Query: what is quantum superposition?


In [ ]:
naive_sbert = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embs = naive_sbert.encode(CORPUS, convert_to_numpy=True)

def naive_rag(user_query: str) -> dict:
    q_emb = naive_sbert.encode([user_query], convert_to_numpy=True)
    corpus_norm = corpus_embs / (np.linalg.norm(corpus_embs, axis=1, keepdims=True) + 1e-9)
    q_norm = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
    sims = (corpus_norm @ q_norm.T).squeeze()
    top_id = int(np.argmax(sims))
    return {"doc_id": top_id, "score": round(float(sims[top_id]), 4), "text": CORPUS[top_id]}

In [ ]:
TEST_QUERIES = [
    "what is quantum entanglement?",
    "challenges in building quantum computers",
    "applications of quantum key distribution",
]

comparison_rows = []

for query in TEST_QUERIES:
    naive_top = naive_rag(query)

    hyp_doc   = hyde_expand(query)
    candidates = retriever.retrieve(hyp_doc, top_k=6)
    adv_top   = rerank(query, candidates, top_k=1)[0]

    different = " Yes" if naive_top["doc_id"] != adv_top["doc_id"] else " No"

    comparison_rows.append({
        "query":        query,
        "naive_top":    f"[{naive_top['doc_id']}] {naive_top['text'][:60]}...",
        "advanced_top": f"[{adv_top['doc_id']}]  {adv_top['text'][:60]}...",
        "different":    different,
    })
    print(f"✔ Done: '{query[:40]}'", flush=True)

print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
for row in comparison_rows:
    print(f"\nQuery   : {row['query']}")
    print(f"Naïve   : {row['naive_top']}")
    print(f"Advanced: {row['advanced_top']}")
    print(f"Different? {row['different']}")

## Part 6 — Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| "how do transformers encode meaning?" | [1] Transformers use self-attention to compute contextual embeddings... | [1] Transformers use self-attention to compute contextual embeddings... | No |
| "optimization techniques for training" | [3] Gradient descent updates model weights by moving in the direction... | [4] Backpropagation computes gradients layer-by-layer using the chain rule... | Yes |
| "what makes RAG better than standard LLMs?" | [11] Retrieval-Augmented Generation (RAG) enhances LLM responses... | [11] Retrieval-Augmented Generation (RAG) enhances LLM responses... | No |

**Observations:**

- **Query 1 & 3:** Both pipelines return the same top doc — the query is clear enough that SBERT alone gets it right.
- **Query 2:** Advanced RAG picks backpropagation over gradient descent, likely because HyDE introduced terms like *chain rule* that shifted retrieval toward a more specific result.
- **Takeaway:** Advanced RAG helps most when the query vocabulary doesn't directly match the corpus.